In [1]:
import numpy as np
import pandas as pd


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df['gender'].value_counts()
df['city'].value_counts()
# gender and coty mei nominal so ohe and age mei ordinal so ordinal encoding



city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [ ]:
df.isnull().sum()
#fever mei 10 values missing hain so fever mei we will use normal imputer

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test,  y_train, test = train_test_split(df.drop(columns='has_covid'), df['has_covid'], test_size=0.2)

In [10]:
X_train

,age,gender,fever,cough,city
9,64,Female,101.0,Mild,Delhi
80,14,Female,99.0,Mild,Mumbai
41,82,Male,NaN,Mild,Kolkata
14,51,Male,104.0,Mild,Bangalore
89,46,Male,103.0,Strong,Bangalore
...,...,...,...,...,...
20,12,Male,98.0,Strong,Bangalore
69,73,Female,103.0,Mild,Delhi
84,69,Female,98.0,Strong,Mumbai
30,15,Male,101.0,Mild,Delhi


WITHOUT COLUMN TRANSFORMER - AAM ZINDAGI

In [ ]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape
# missing valyes ko mean se replace kra

(80, 1)

In [13]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [19]:
ohe = OneHotEncoder(drop = 'first', sparse_output = False) # we do sparse false to get array output instead of sparse matrix
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])
X_train_gender_city.shape


(80, 4)

In [21]:
#ab age kko alag kr denge, and then us new column ko baaki saare transformed columns ke sath concatonate kr denge
X_train_age = X_train.drop(columns=['gender', 'city', 'cough', 'fever']).values
X_test_age = X_test.drop(columns=['gender', 'city', 'cough', 'fever']).values

X_train_age.shape

(80, 1)

In [24]:
X_train_transformed =np.concatenate([X_train_age, X_train_fever, X_train_cough, X_train_gender_city], axis=1)
X_test_transformed =np.concatenate([X_test_age, X_test_fever, X_test_cough, X_test_gender_city], axis=1)

X_train_transformed.shape

(80, 7)

COLUMN TRANSFORMER - MENTOS ZINDAGI

In [26]:
from sklearn.compose import ColumnTransformer



In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']), ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']), ('tnf3', OneHotEncoder(drop = 'first', sparse_output=False), ['gender', 'city'])] , remainder='passthrough')

#remainder can either be passthrough or drop, if we do passthrough then the columns which are not mentioned in the transformers list will be passed through without any transformation, and if we do drop then the columns which are not mentioned in the transformers list will be dropped
transformer.fit_transform(X_train)
transformer.fit_transform(X_test)


(20, 7)